# Official filing identity and alias checks
The default registry contains only official filing identities. Trial counts are not an input or completion criterion. Run from the project root; source code must be installed or available on PYTHONPATH.

In [ ]:
import runpy
checks = runpy.run_path('scripts/check_official_aliases.py')
checks['check']()

In [ ]:
from cn_hospital_aliases import HospitalRegistry
registry = HospitalRegistry.load_default()
[(h.canonical_name, len(h.aliases), h.filing_number) for h in registry.hospitals if h.aliases][:10]

## Frozen review queue
This checks name/location groups, not distinct hospitals or distinct trials. Reconciliation does not perform a new manual source review. Reference-only names must not receive a confirmed identity.

In [ ]:
import csv, hashlib, json
from pathlib import Path
directory = Path('data/official')
summary = json.loads((directory / 'trial_site_verification_summary.json').read_text())
assert hashlib.sha256((directory / 'trial_site_review_baseline.csv').read_bytes()).hexdigest() == summary['baseline_sha256']
with (directory / 'trial_site_verification_register.csv').open(encoding='utf-8-sig') as handle:
    rows = list(csv.DictReader(handle))
assert len(rows) == summary['rows_checked_against_available_evidence']
assert all(not row['official_hospital_id'] for row in rows if row['location_verification_status'] != 'passes_available_location_filters')
assert all(row['approved_name_evidence_json'] != '[]' for row in rows if row['name_verification_status'] == 'verified_name_relation')
print(summary)

In [ ]:
with (directory / 'institution_confirmation_register.csv').open(encoding='utf-8-sig') as handle:
    identities = list(csv.DictReader(handle))
assert len(identities) == len(registry.hospitals)
assert {row['hospital_id'] for row in identities} == {h.hospital_id for h in registry.hospitals}
assert all(row['english_review_status'] == 'deferred_not_required_for_identity' for row in identities if not row['english_name'])
decisions = json.loads(Path('data/curated/chinese_name_decisions.json').read_text())['decisions']
for decision in decisions:
    if decision['decision'] == 'approve_parallel_hospital_name':
        assert any(m.hospital.hospital_id == decision['hospital_id'] for m in registry.resolve(decision['candidate_alias']))
print('All filing identities retained; Chinese decisions applied; English remains optional')
